In [1]:
import pandas as pd
import numpy as np
from google.colab import files
from IPython.display import display

In [3]:
uploaded = files.upload()
print("\nUploaded files:")
for filename in uploaded.keys():
    print("-", filename)

Saving MCQ Question Extraction Dataset (1).xlsx to MCQ Question Extraction Dataset (1).xlsx
Saving Textbook Essay and Text-Based Questions.xlsx to Textbook Essay and Text-Based Questions.xlsx

Uploaded files:
- MCQ Question Extraction Dataset (1).xlsx
- Textbook Essay and Text-Based Questions.xlsx


In [4]:
dataframes = {}
for filename in uploaded.keys():
    if filename.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(filename)
    elif filename.lower().endswith(".csv"):
        df = pd.read_csv(filename)
    else:
        print(f"Skipped unsupported file: {filename}")
        continue
    dataframes[filename] = df
    print(f"\n{'=' * 60}")
    print("File:", filename)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))



File: MCQ Question Extraction Dataset (1).xlsx
Shape: (168, 15)
Columns: ['question_id', 'question_text', 'option_A', 'option_B', 'option_C', 'option_D', 'correct_option', 'correct_answer', 'chapter', 'lesson', 'page_number', 'answer_source', 'verification_status', 'source_file', 'Source']


,question_id,question_text,option_A,option_B,option_C,option_D,correct_option,correct_answer,chapter,lesson,page_number,answer_source,verification_status,source_file,Source
0,MCQ-001,Which of the following options represents the ...,The emergence of smartphones,The widespread use of personal computers (PCs),The spread of cloud computing,The availability of the Internet for commercia...,B,The widespread use of personal computers (PCs),Unit One: Information Technology and Society,Lesson 1-1 Development of Information Technolo...,3,Book Content,Solved by AI,Programming-ArtificialIntelligence-En-EB-Asses...,"1, 2"
1,MCQ-002,What is the empirical observation that describ...,Cloud computing,Moore’s Law,Edge computing,Quantum superposition,B,Moore’s Law,Unit One: Information Technology and Society,Lesson 1-1 Development of Information Technolo...,3,Book Content,Solved by AI,Programming-ArtificialIntelligence-En-EB-Asses...,"1, 2"
2,MCQ-003,What term refers to a working style in which a...,Online Learning,E-commerce,Remote Work,Cashless Payment,C,Remote Work,Unit One: Information Technology and Society,Lesson 1-1 Development of Information Technolo...,4,Book Content,Solved by AI,Programming-ArtificialIntelligence-En-EB-Asses...,"1, 2"



File: Textbook Essay and Text-Based Questions.xlsx
Shape: (149, 11)
Columns: ['question_id', 'question_text', 'model_answer', 'chapter', 'lesson', 'page_number', 'source_file', 'answer_source', 'verification_status', 'Source', 'Unnamed: 10']


,question_id,question_text,model_answer,chapter,lesson,page_number,source_file,answer_source,verification_status,Source,Unnamed: 10
0,TEXT-001,How has information technology developed throu...,Information technology advanced through a seri...,1 Information Technology and Society,1-1 Development of Information Technology and ...,"7, 26",Programming-ArtificialIntelligence-En-EB-part1...,Book Content,Verified from Source,1,NaN
1,TEXT-002,"In autonomous driving, why is it necessary to ...","In autonomous driving, because a delay of even...",1 Information Technology and Society,1-1 Development of Information Technology and ...,14,Programming-ArtificialIntelligence-En-EB-part1...,Book Content,Verified from Source,1,NaN
2,TEXT-003,Analyze how the spread of cloud computing (fro...,The spread of cloud computing from the 2010s o...,1 Information Technology and Society,1-1 Development of Information Technology and ...,"9, 23",Programming-ArtificialIntelligence-En-EB-part1...,Book Content,Verified from Source,1,NaN


In [5]:
mcq = dataframes["MCQ Question Extraction Dataset (1).xlsx"].copy()
essay = dataframes["Textbook Essay and Text-Based Questions.xlsx"].copy()
# Remove accidental spaces from column names
mcq.columns = mcq.columns.astype(str).str.strip()
essay.columns = essay.columns.astype(str).str.strip()
# Remove completely empty columns only
mcq = mcq.dropna(axis=1, how="all")
essay = essay.dropna(axis=1, how="all")
# Remove accidental whitespace from text cells
def clean_text_cells(df):
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].map(
            lambda x: x.strip() if isinstance(x, str) else x
        )
    return df
mcq = clean_text_cells(mcq)
essay = clean_text_cells(essay)
# Rename essay answer column to match MCQ
essay = essay.rename(columns={"model_answer": "correct_answer"})
# Mark each question type
mcq["question_type"] = "MCQ"
essay["question_type"] = "Essay"
print("MCQ shape:", mcq.shape)
print("Essay shape:", essay.shape)
print("\nMCQ columns:")
print(mcq.columns.tolist())
print("\nEssay columns:")
print(essay.columns.tolist())

MCQ shape: (168, 16)
Essay shape: (149, 12)

MCQ columns:
['question_id', 'question_text', 'option_A', 'option_B', 'option_C', 'option_D', 'correct_option', 'correct_answer', 'chapter', 'lesson', 'page_number', 'answer_source', 'verification_status', 'source_file', 'Source', 'question_type']

Essay columns:
['question_id', 'question_text', 'correct_answer', 'chapter', 'lesson', 'page_number', 'source_file', 'answer_source', 'verification_status', 'Source', 'Unnamed: 10', 'question_type']


In [6]:

# Standard schema shared by both question types
common_columns = [
    "question_id",
    "question_type",
    "question_text",
    "option_A",
    "option_B",
    "option_C",
    "option_D",
    "correct_option",
    "correct_answer",
    "chapter",
    "lesson",
    "page_number",
    "source_file",
    "answer_source",
    "verification_status",
    "Source",
]

# Add missing columns without overwriting existing values
for col in common_columns:
    if col not in mcq.columns:
        mcq[col] = pd.NA
    if col not in essay.columns:
        essay[col] = pd.NA

# Keep the same column order
mcq = mcq[common_columns]
essay = essay[common_columns]

# Combine rows
questions = pd.concat([mcq, essay], ignore_index=True)

print("Combined dataset shape:", questions.shape)
print("\nQuestion types:")
print(questions["question_type"].value_counts(dropna=False))

display(questions.head(3))
display(questions.tail(3))

Combined dataset shape: (317, 16)

Question types:
question_type
MCQ      168
Essay    149
Name: count, dtype: int64


,question_id,question_type,question_text,option_A,option_B,option_C,option_D,correct_option,correct_answer,chapter,lesson,page_number,source_file,answer_source,verification_status,Source
0,MCQ-001,MCQ,Which of the following options represents the ...,The emergence of smartphones,The widespread use of personal computers (PCs),The spread of cloud computing,The availability of the Internet for commercia...,B,The widespread use of personal computers (PCs),Unit One: Information Technology and Society,Lesson 1-1 Development of Information Technolo...,3,Programming-ArtificialIntelligence-En-EB-Asses...,Book Content,Solved by AI,"1, 2"
1,MCQ-002,MCQ,What is the empirical observation that describ...,Cloud computing,Moore’s Law,Edge computing,Quantum superposition,B,Moore’s Law,Unit One: Information Technology and Society,Lesson 1-1 Development of Information Technolo...,3,Programming-ArtificialIntelligence-En-EB-Asses...,Book Content,Solved by AI,"1, 2"
2,MCQ-003,MCQ,What term refers to a working style in which a...,Online Learning,E-commerce,Remote Work,Cashless Payment,C,Remote Work,Unit One: Information Technology and Society,Lesson 1-1 Development of Information Technolo...,4,Programming-ArtificialIntelligence-En-EB-Asses...,Book Content,Solved by AI,"1, 2"


,question_id,question_type,question_text,option_A,option_B,option_C,option_D,correct_option,correct_answer,chapter,lesson,page_number,source_file,answer_source,verification_status,Source
314,TEXT-147,Essay,How do the curriculum chapters connect with ea...,NaN,NaN,NaN,NaN,NaN,They work together like building a smart educa...,Curriculum Structure,Curriculum Integration,Not in source,منهج البرمجة تانية ثانوي بكالوريا 2026/2027 \,كتاب الوزارة عربي ولغات,Book Content,Verified from Source
315,TEXT-148,Essay,Is the 2nd secondary programming curriculum th...,NaN,NaN,NaN,NaN,NaN,It contains both: the theoretical part helps u...,Curriculum Structure,Theory vs Practice,Not in source,منهج البرمجة تانية ثانوي بكالوريا 2026/2027 \,كتاب الوزارة عربي ولغات,Book Content,Verified from Source
316,TEXT-149,Essay,How should a student prepare for the 2nd secon...,NaN,NaN,NaN,NaN,NaN,Review computer and internet basics. 2. Review...,Preparation,Preparation Steps,Not in source,منهج البرمجة تانية ثانوي بكالوريا 2026/2027 \,كتاب الوزارة عربي ولغات,Book Content,Verified from Source


In [8]:

# Normalize question text for duplicate detection
questions["question_normalized"] = (
    questions["question_text"]
    .astype("string")
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.rstrip("?.! ")
)

# Find potential duplicates within the same question type
duplicate_mask = questions.duplicated(
    subset=["question_type", "question_normalized"],
    keep=False
)

# Select review columns and sort safely
duplicates_review = questions.loc[
    duplicate_mask,
    [
        "question_id",
        "question_type",
        "question_text",
        "chapter",
        "lesson",
        "source_file",
        "question_normalized",
    ]
].sort_values(["question_type", "question_normalized"])

print("Potential duplicate rows:", int(duplicate_mask.sum()))
display(duplicates_review)


Potential duplicate rows: 0


,question_id,question_type,question_text,chapter,lesson,source_file,question_normalized


In [9]:

# Preserve original IDs for traceability
questions["original_question_id"] = questions["question_id"]

# Generate a unique ID for every row
questions["question_id"] = [
    f"Q-{i:04d}" for i in range(1, len(questions) + 1)
]

# Keep the original source label but remove helper column
questions = questions.drop(columns=["question_normalized"])

# Final quality checks
assert questions["question_id"].is_unique
assert len(questions) == len(mcq) + len(essay)

print("Final shape:", questions.shape)
print("Unique IDs:", questions["question_id"].is_unique)
print(questions["question_type"].value_counts())

# Save both the merged data and a review report
with pd.ExcelWriter("Question_Bank_Cleaned.xlsx", engine="openpyxl") as writer:
    questions.to_excel(writer, sheet_name="All_Questions", index=False)
    duplicates_review.to_excel(writer, sheet_name="Duplicates_Review", index=False)
    missing_report.assign(
        question_id=questions["question_id"]
    ).to_excel(writer, sheet_name="Missing_Data_Check", index=False)

files.download("Question_Bank_Cleaned.xlsx")


Final shape: (317, 17)
Unique IDs: True
question_type
MCQ      168
Essay    149
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:

# 1. Check that question IDs are unique
print("Total questions:", len(questions))
print("Unique question IDs:", questions["question_id"].nunique())

# 2. Check required fields for MCQs
mcq_data = questions[questions["question_type"] == "MCQ"]

mcq_missing_options = mcq_data[
    mcq_data[["option_A", "option_B", "option_C", "option_D"]]
    .isna().any(axis=1)
]

mcq_missing_correct_option = mcq_data[
    mcq_data["correct_option"].isna()
    | mcq_data["correct_option"].astype("string").str.strip().eq("")
]

# 3. Check missing model answers for Essay questions
essay_data = questions[questions["question_type"] == "Essay"]

essay_missing_answers = essay_data[
    essay_data["correct_answer"].isna()
    | essay_data["correct_answer"].astype("string").str.strip().eq("")
]

# 4. Display the report
print("\nMCQs missing one or more options:", len(mcq_missing_options))
print("MCQs missing correct option:", len(mcq_missing_correct_option))
print("Essays missing model answers:", len(essay_missing_answers))

display(mcq_missing_options)
display(mcq_missing_correct_option)
display(essay_missing_answers)

Total questions: 317
Unique question IDs: 317

MCQs missing one or more options: 0
MCQs missing correct option: 0
Essays missing model answers: 0


,question_id,question_type,question_text,option_A,option_B,option_C,option_D,correct_option,correct_answer,chapter,lesson,page_number,source_file,answer_source,verification_status,Source,original_question_id


,question_id,question_type,question_text,option_A,option_B,option_C,option_D,correct_option,correct_answer,chapter,lesson,page_number,source_file,answer_source,verification_status,Source,original_question_id


,question_id,question_type,question_text,option_A,option_B,option_C,option_D,correct_option,correct_answer,chapter,lesson,page_number,source_file,answer_source,verification_status,Source,original_question_id
